In [9]:
!pip install -q "datasets>=2.20" "tokenizers>=0.20,<1.0" numpy tqdm


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import json
import random
import time
import re
import html
import unicodedata

from pathlib import Path
from collections import Counter

import numpy as np

from datasets import load_dataset
from tokenizers import (
    Tokenizer,
    models,
    pre_tokenizers,
    decoders,
    trainers
)

from tqdm.auto import tqdm

print("numpy      :", np.__version__)
print("Tokenizer  :", __import__("tokenizers").__version__)

numpy      : 2.5.3
Tokenizer  : 0.23.2


In [12]:
PROJECT_ROOT = Path(r"C:\Users\Rasulbekk\Desktop\my_LLM")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FILTERED_DIR = PROJECT_ROOT / "data" / "filtered"
FINAL_DIR = PROJECT_ROOT / "data" / "final"

TOKENIZER_DIR = PROJECT_ROOT / "tokenizer"
DATASET_DIR = PROJECT_ROOT / "dataset"

FINAL_FILE = FINAL_DIR / "tinystories_final.jsonl"
TOKENIZER_FILE = TOKENIZER_DIR / "tokenizer.json"

TRAIN_BIN = DATASET_DIR / "train.bin"
VAL_BIN = DATASET_DIR / "val.bin"
META_FILE = DATASET_DIR / "meta.json"


# Papkalarni yaratish
for directory in [
    RAW_DIR,
    CLEANED_DIR,
    FILTERED_DIR,
    FINAL_DIR,
    TOKENIZER_DIR,
    DATASET_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


print("=" * 60)
print("PROJECT PATHS")
print("=" * 60)

print("Project root :", PROJECT_ROOT)
print("Raw data     :", RAW_DIR)
print("Cleaned data :", CLEANED_DIR)
print("Filtered data:", FILTERED_DIR)
print("Final corpus :", FINAL_FILE)
print("Tokenizer    :", TOKENIZER_FILE)
print("Dataset      :", DATASET_DIR)

PROJECT PATHS
Project root : C:\Users\Rasulbekk\Desktop\my_LLM
Raw data     : C:\Users\Rasulbekk\Desktop\my_LLM\data\raw
Cleaned data : C:\Users\Rasulbekk\Desktop\my_LLM\data\cleaned
Filtered data: C:\Users\Rasulbekk\Desktop\my_LLM\data\filtered
Final corpus : C:\Users\Rasulbekk\Desktop\my_LLM\data\final\tinystories_final.jsonl
Tokenizer    : C:\Users\Rasulbekk\Desktop\my_LLM\tokenizer\tokenizer.json
Dataset      : C:\Users\Rasulbekk\Desktop\my_LLM\dataset


In [14]:
dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train"
)

print(dataset)
print("Columns:", dataset.column_names)
print("Documents:", len(dataset))

print("\nFirst document:")
print(dataset[0]["text"][:500])

c:\Users\Rasulbekk\Desktop\my_LLM\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rasulbekk\.cache\huggingface\hub\datasets--roneneldan--TinyStories. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 21990/21990 [00:00<00:00, 259921.84 examples/s]


Dataset({
    features: ['text'],
    num_rows: 2119719
})
Columns: ['text']
Documents: 2119719

First document:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


# Datasetni saqlash

In [16]:
RAW_FILE = RAW_DIR / "tinystories_train.jsonl"

with open(RAW_FILE, "w", encoding="utf-8") as f:

    for item in tqdm(dataset, desc="Saving raw data"):
        record = {
            "text": item["text"]
        }

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print("\nRaw dataset saqlandi:")
print(RAW_FILE)

print(
    "Size:",
    round(RAW_FILE.stat().st_size / 1024**2, 2),
    "MB"
)

Saving raw data: 100%|██████████| 2119719/2119719 [01:15<00:00, 28156.39it/s]


Raw dataset saqlandi:
C:\Users\Rasulbekk\Desktop\my_LLM\data\raw\tinystories_train.jsonl
Size: 1870.22 MB


# Raw datasetni o‘qish

In [17]:
RAW_FILE = RAW_DIR / "tinystories_train.jsonl"

raw_docs = []

with open(RAW_FILE, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Loading raw data"):
        raw_docs.append(json.loads(line))

print("Loaded documents:", len(raw_docs))
print("First document:")
print(raw_docs[0]["text"][:500])

Loading raw data: 2119719it [00:21, 97244.37it/s] 

Loaded documents: 2119719
First document:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


# Cleaning funksiyasi

In [18]:
import re

def clean_text(text):
    # Ortiqcha bo'sh joylarni olib tashlash
    text = re.sub(r"\s+", " ", text)

    # Boshi va oxiridagi bo'sh joylarni olib tashlash
    text = text.strip()

    return text

# Barcha documentlarni cleaning qilish

In [19]:
cleaned_docs = []

for doc in tqdm(raw_docs, desc="Cleaning data"):
    text = clean_text(doc["text"])

    if text:
        cleaned_docs.append({"text": text})

print("Raw documents:", len(raw_docs))
print("Cleaned documents:", len(cleaned_docs))

Cleaning data: 100%|██████████| 2119719/2119719 [02:01<00:00, 17479.09it/s]

Raw documents: 2119719
Cleaned documents: 2119489


#  Cleaned datasetni saqlash

In [20]:
CLEANED_FILE = CLEANED_DIR / "tinystories_cleaned.jsonl"

with open(CLEANED_FILE, "w", encoding="utf-8") as f:
    for doc in tqdm(cleaned_docs, desc="Saving cleaned data"):
        f.write(
            json.dumps(
                doc,
                ensure_ascii=False
            ) + "\n"
        )

print("Cleaned dataset saqlandi:")
print(CLEANED_FILE)
print(
    "Size:",
    round(CLEANED_FILE.stat().st_size / 1024**2, 2),
    "MB"
)

Saving cleaned data: 100%|██████████| 2119489/2119489 [00:30<00:00, 69786.20it/s]

Cleaned dataset saqlandi:
C:\Users\Rasulbekk\Desktop\my_LLM\data\cleaned\tinystories_cleaned.jsonl
Size: 1842.19 MB


# Datasetni filter qilish

In [21]:
MIN_CHARS = 50
MAX_CHARS = 10000

filtered_docs = []

for doc in tqdm(cleaned_docs, desc="Filtering data"):
    text = doc["text"]

    if MIN_CHARS <= len(text) <= MAX_CHARS:
        filtered_docs.append({"text": text})

print("Cleaned documents:", len(cleaned_docs))
print("Filtered documents:", len(filtered_docs))
print("Removed:", len(cleaned_docs) - len(filtered_docs))

Filtering data: 100%|██████████| 2119489/2119489 [00:01<00:00, 1259575.80it/s]

Cleaned documents: 2119489
Filtered documents: 2119482
Removed: 7


# Filter natijasini tekshirish

In [22]:
print("First filtered document:")
print(filtered_docs[0]["text"])

print("\nCharacter count:")
print(len(filtered_docs[0]["text"]))

First filtered document:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt. Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt." Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.

Character count:
699


# Final datasetni saqlash

In [23]:
FINAL_FILE = FINAL_DIR / "tinystories_final.jsonl"

with open(FINAL_FILE, "w", encoding="utf-8") as f:
    for doc in tqdm(filtered_docs, desc="Saving final dataset"):
        f.write(
            json.dumps(
                doc,
                ensure_ascii=False
            ) + "\n"
        )

print("\nFinal dataset saqlandi:")
print(FINAL_FILE)

print(
    "Size:",
    round(FINAL_FILE.stat().st_size / 1024**2, 2),
    "MB"
)

Saving final dataset: 100%|██████████| 2119482/2119482 [00:30<00:00, 70197.09it/s]


Final dataset saqlandi:
C:\Users\Rasulbekk\Desktop\my_LLM\data\final\tinystories_final.jsonl
Size: 1842.19 MB


# Final datasetni tekshirish

In [24]:
print("Final file exists:", FINAL_FILE.exists())
print("Final file size:",
      round(FINAL_FILE.stat().st_size / 1024**2, 2),
      "MB")

Final file exists: True
Final file size: 1842.19 MB


# Tokenizer sozlamalari

# ByteLevel BPE tokenizer

In [25]:
VOCAB_SIZE = 8000
MIN_FREQUENCY = 2
SPECIAL_TOKENS = ["<eos>"]
TOKENIZER_SAMPLE_DOCS = 200_000

# Tokenizer uchun text generator

In [26]:
def text_iterator(docs, limit=None):
    count = 0

    for doc in docs:
        yield doc["text"]

        count += 1

        if limit is not None and count >= limit:
            break

# Tokenizer yaratish

In [27]:
from tokenizers import Tokenizer
from tokenizers import models
from tokenizers import pre_tokenizers
from tokenizers import decoders
from tokenizers.trainers import BpeTrainer

tokenizer = Tokenizer(models.BPE())

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(
    add_prefix_space=False
)

tokenizer.decoder = decoders.ByteLevel()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)

# Tokenizerni train qilish

In [28]:
tokenizer.train_from_iterator(
    text_iterator(
        filtered_docs,
        limit=TOKENIZER_SAMPLE_DOCS
    ),
    trainer=trainer
)

print("Tokenizer training tugadi.")
print("Vocab size:", tokenizer.get_vocab_size())

Tokenizer training tugadi.
Vocab size: 8000


# Tokenizerni saqlash

In [29]:
TOKENIZER_FILE = TOKENIZER_DIR / "tokenizer.json"

tokenizer.save(str(TOKENIZER_FILE))

print("Tokenizer saqlandi:")
print(TOKENIZER_FILE)

Tokenizer saqlandi:
C:\Users\Rasulbekk\Desktop\my_LLM\tokenizer\tokenizer.json


# Dataset sozlamalari

In [30]:
BLOCK_SIZE = 256
VAL_RATIO = 0.01
SEED = 1337

print("BLOCK_SIZE:", BLOCK_SIZE)
print("VAL_RATIO:", VAL_RATIO)
print("SEED:", SEED)

BLOCK_SIZE: 256
VAL_RATIO: 0.01
SEED: 1337


# Final datasetni qayta yuklash

In [31]:
docs = []

with open(FINAL_FILE, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Loading final dataset"):
        docs.append(json.loads(line))

print("Final documents:", len(docs))

Loading final dataset: 2119482it [00:21, 97574.08it/s] 

Final documents: 2119482


# Train / Validation ga ajratish

In [32]:
import random

random.seed(SEED)

indices = list(range(len(docs)))
random.shuffle(indices)

val_size = int(len(docs) * VAL_RATIO)

val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_docs = [docs[i] for i in train_indices]
val_docs = [docs[i] for i in val_indices]

print("Train documents:", len(train_docs))
print("Validation documents:", len(val_docs))

Train documents: 2098288
Validation documents: 21194


# Token ID chegarasini tekshirish

In [33]:
vocab_size = tokenizer.get_vocab_size()

print("Tokenizer vocab size:", vocab_size)
print("uint16 max:", 65535)

assert vocab_size <= 65535, \
    "Vocabulary uint16 uchun juda katta!"

print("uint16 uchun OK")

Tokenizer vocab size: 8000
uint16 max: 65535
uint16 uchun OK


# biz <eos> token ID sini olish

In [34]:
EOS_ID = tokenizer.token_to_id("<eos>")

print("EOS token ID:", EOS_ID)

assert EOS_ID is not None, \
    "<eos> tokenizer vocabulary ichida topilmadi!"

EOS token ID: 0


# Documentlarni tokenlarga aylantirish

In [35]:
import numpy as np

def encode_documents(documents):
    all_tokens = []

    for doc in tqdm(documents, desc="Encoding documents"):
        text = doc["text"]

        encoded = tokenizer.encode(text)
        tokens = encoded.ids

        tokens.append(EOS_ID)
        all_tokens.extend(tokens)

    return np.array(all_tokens, dtype=np.uint16)

# Encode funksiyasi

In [36]:
train_tokens = encode_documents(train_docs)

print("Train tokens:", len(train_tokens))
print("Train dtype:", train_tokens.dtype)
print("Train size:",
      round(train_tokens.nbytes / 1024**2, 2),
      "MB")

Encoding documents: 100%|██████████| 2098288/2098288 [17:41<00:00, 1975.98it/s]


Train tokens: 443762028
Train dtype: uint16
Train size: 846.41 MB


# Validation tokenlarini yaratish

In [37]:
val_tokens = encode_documents(val_docs)

print("Validation tokens:", len(val_tokens))
print("Validation dtype:", val_tokens.dtype)
print(
    "Validation size:",
    round(val_tokens.nbytes / 1024**2, 2),
    "MB"
)

Encoding documents: 100%|██████████| 21194/21194 [00:16<00:00, 1279.83it/s]

Validation tokens: 4452398
Validation dtype: uint16
Validation size: 8.49 MB


# train.bin saqlash

In [38]:
TRAIN_BIN = DATASET_DIR / "train.bin"

train_tokens.tofile(TRAIN_BIN)

print("train.bin saqlandi:")
print(TRAIN_BIN)

print(
    "Size:",
    round(TRAIN_BIN.stat().st_size / 1024**2, 2),
    "MB"
)

train.bin saqlandi:
C:\Users\Rasulbekk\Desktop\my_LLM\dataset\train.bin
Size: 846.41 MB


# val.bin saqlash

In [39]:
VAL_BIN = DATASET_DIR / "val.bin"

val_tokens.tofile(VAL_BIN)

print("val.bin saqlandi:")
print(VAL_BIN)

print(
    "Size:",
    round(VAL_BIN.stat().st_size / 1024**2, 2),
    "MB"
)

val.bin saqlandi:
C:\Users\Rasulbekk\Desktop\my_LLM\dataset\val.bin
Size: 8.49 MB


# meta.json

In [40]:
META_FILE = DATASET_DIR / "meta.json"

meta = {
    "vocab_size": tokenizer.get_vocab_size(),
    "block_size": BLOCK_SIZE,
    "train_tokens": int(len(train_tokens)),
    "val_tokens": int(len(val_tokens)),
    "train_documents": len(train_docs),
    "val_documents": len(val_docs),
    "val_ratio": VAL_RATIO,
    "seed": SEED,
    "dtype": "uint16",
    "eos_token": "<eos>",
    "eos_token_id": int(EOS_ID)
}

with open(META_FILE, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("meta.json saqlandi:")
print(META_FILE)

meta.json saqlandi:
C:\Users\Rasulbekk\Desktop\my_LLM\dataset\meta.json


# Yakuniy tekshiruv

In [41]:
print("=" * 60)
print("FINAL DATASET CHECK")
print("=" * 60)

for file in [FINAL_FILE, TOKENIZER_FILE, TRAIN_BIN, VAL_BIN, META_FILE]:
    print(
        f"{file.name:25} ->",
        "OK" if file.exists() else "MISSING"
    )

FINAL DATASET CHECK
tinystories_final.jsonl   -> OK
tokenizer.json            -> OK
train.bin                 -> OK
val.bin                   -> OK
meta.json                 -> OK


my_LLM/
├── data/
│   └── final/
│       └── tinystories_final.jsonl   ✅
│
├── tokenizer/
│   └── tokenizer.json                ✅
│
└── dataset/
    ├── experiment.ipynb              ✅
    ├── train.bin                     ✅
    ├── val.bin                       ✅
    └── meta.json                     ✅

# Dataset Loader

In [42]:
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path(r"C:\Users\Rasulbekk\Desktop\my_LLM")

DATASET_DIR = PROJECT_ROOT / "dataset"

TRAIN_BIN = DATASET_DIR / "train.bin"
VAL_BIN = DATASET_DIR / "val.bin"

BLOCK_SIZE = 256

# train.binni tekshirish

In [43]:
train_data = np.memmap(
    TRAIN_BIN,
    dtype=np.uint16,
    mode="r"
)

val_data = np.memmap(
    VAL_BIN,
    dtype=np.uint16,
    mode="r"
)

print("Train tokens:", len(train_data))
print("Val tokens:", len(val_data))
print("Train dtype:", train_data.dtype)
print("Val dtype:", val_data.dtype)

Train tokens: 443762028
Val tokens: 4452398
Train dtype: uint16
Val dtype: uint16


In [46]:
!pip install torch

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.8/124.1 MB 4.2 MB/s eta 0:00:30
   ---------------------------------------- 1.3/124.1 MB 4.9 MB/s eta 0:00:26
    --------------------------------------- 1.8/124.1 MB 3.5 MB/s eta 0:00:35
   - -------------------------------------- 3.1/124.1 MB 3.8 MB/s eta 0:00:33
   - -------------------------------------- 4.5/124.1 MB 4.3 MB/s eta 0:00:28
   - -------------------------------------- 5.2/124.1 MB 4.5 MB/s eta 0:00:27
   -- ------------------------------------- 6.6/124.1 MB 4.4 


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Batch olish funksiyasi

In [47]:
import torch

def get_batch(data, batch_size=8, block_size=256):
    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        torch.from_numpy(
            data[i:i + block_size].astype(np.int64)
        )
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(
            data[i + 1:i + block_size + 1].astype(np.int64)
        )
        for i in ix
    ])

    return x, y

# Batchni test qilish

In [48]:
BATCH_SIZE = 8

x, y = get_batch(
    train_data,
    batch_size=BATCH_SIZE,
    block_size=BLOCK_SIZE
)

print("X shape:", x.shape)
print("Y shape:", y.shape)
print("X dtype:", x.dtype)
print("Y dtype:", y.dtype)

X shape: torch.Size([8, 256])
Y shape: torch.Size([8, 256])
X dtype: torch.int64
Y dtype: torch.int64


# Dataset → Model pipeline tayyor. ✅